<div dir="rtl" align="right">

# نسبُ الطاقةِ بينَ النطاقاتِ

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نَحسبُ نسبَ الطاقةِ بينَ النطاقاتِ التردديّةِ كَمُؤشّراتٍ للحالةِ الذهنيّة: ثيتا/ألفا لِلإجهاد، ألفا/بيتا لِلاسترخاء، وثيتا/بيتا لِلانتباه.

## المُخرجاتُ المُتوقّعةُ

- ثلاثةُ مخططاتٍ لِلنسبِ الثلاثِ على الزمنِ
- خطٌّ أفقيٌّ مُتقطّعٌ يُمثّلُ المتوسطَ لِكلِّ نسبةٍ
- تَذبذبٌ يَعكسُ تَغيّرَ الحالةِ الذهنيّةِ عبرَ الزمن

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| WINDOW | 1000 | نافذةُ 5 ثوانٍ |
| STEP | 500 | خطوةُ 2.5 ثانيةٍ |
| النطاقاتُ | 4 | دلتا، ثيتا، ألفا، بيتا |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly mne wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. حسابُ نسبِ الطاقةِ

نَحسبُ طاقةَ كلِّ نطاقٍ على نوافذَ مُتحرّكةٍ، ثمّ نَحسبُ النسبَ الثلاثَ.

</div>

In [ ]:
from scipy.signal import welch

WINDOW = 1000
STEP = 500
BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
}

def compute_band_powers(segment, fs):
    freqs, psd = welch(segment, fs=fs, nperseg=1024)
    powers = {}
    for name, (fmin, fmax) in BANDS.items():
        mask = (freqs >= fmin) & (freqs <= fmax)
        powers[name] = np.trapezoid(psd[mask], freqs[mask])
    return powers

channel_data = eeg_data[:, 0]
n_windows = (len(channel_data) - WINDOW) // STEP + 1
theta_alpha = []
alpha_beta = []
theta_beta = []
window_centers = []

for i in range(n_windows):
    start = i * STEP
    end = start + WINDOW
    segment = channel_data[start:end]
    powers = compute_band_powers(segment, fs)
    theta_alpha.append(powers['theta'] / powers['alpha'])
    alpha_beta.append(powers['alpha'] / powers['beta'])
    theta_beta.append(powers['theta'] / powers['beta'])
    window_centers.append((start + end) / 2 / fs)

theta_alpha = np.array(theta_alpha)
alpha_beta = np.array(alpha_beta)
theta_beta = np.array(theta_beta)
print(f'Computed {n_windows} windows')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- النوافذُ التي تَنحرفُ عن المتوسطِ تُشيرُ إلى تَغيّرٍ في الحالةِ الذهنيّة
- ارتفاعُ ثيتا/ألفا يَدلُّ على إجهادٍ، وانخفاضُهُ على استرخاء
- ارتفاعُ ألفا/بيتا يَدلُّ على استرخاء، وانخفاضُهُ على انتباه


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Stress indicator (Theta/Alpha)',
                                    'Relaxation indicator (Alpha/Beta)',
                                    'Attention indicator (Theta/Beta)'))
fig.add_trace(go.Scatter(x=window_centers, y=theta_alpha, name='Theta/Alpha',
                         line=dict(color='red', width=1.5)), row=1, col=1)
fig.add_hline(y=np.mean(theta_alpha), line_dash='dash', line_color='black', row=1, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=alpha_beta, name='Alpha/Beta',
                         line=dict(color='green', width=1.5)), row=2, col=1)
fig.add_hline(y=np.mean(alpha_beta), line_dash='dash', line_color='black', row=2, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=theta_beta, name='Theta/Beta',
                         line=dict(color='blue', width=1.5)), row=3, col=1)
fig.add_hline(y=np.mean(theta_beta), line_dash='dash', line_color='black', row=3, col=1)
fig.update_layout(height=900, title_text='Band Power Ratios - Mental State Indicators',
                  xaxis3_title='Time (s)', showlegend=True)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- نسبُ الطاقةِ مُؤشّراتٌ بسيطةٌ وفعّالةٌ لِلحالةِ الذهنيّة
- ثيتا/ألفا لِلإجهاد، ألفا/بيتا لِلاسترخاء، ثيتا/بيتا لِلانتباه
- تُستخدمُ كَقيمٍ أساسيّةٍ لِكلِّ فردٍ في الممارسةِ العمليّة
- لا تَتطلّبُ تعلّمَ الآلةِ لكنّها أقلُّ دقّةً في المهامِّ المُعقّدة


</div>